In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/loan_prediction.csv')

# Display the first 5 rows of the DataFrame
display(df.head())

In [ ]:
# Display general information about the DataFrame, including data types and non-null values
display(df.info())

In [ ]:
# Calculate the percentage of missing values for each column
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

# Display columns with missing values and their percentages, sorted in descending order
missing_info = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
})
missing_info = missing_info[missing_info['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False)

display(missing_info)

In [ ]:
# Impute missing values

# Categorical columns: fill with mode
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Married'] = df['Married'].fillna(df['Married'].mode()[0])
df['Dependents'] = df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed'] = df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mode()[0])
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0])

# Numerical columns: fill with median (robust to outliers)
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())

# Verify that there are no more missing values
print('Missing values after imputation:')
display(df.isnull().sum())

In [ ]:
# Identify categorical columns (object type)
categorical_cols = df.select_dtypes(include='object').columns

print('Unique values and their counts for categorical columns:')
for col in categorical_cols:
    print(f"\nColumn: {col}")
    display(df[col].value_counts())

In [ ]:
# Drop Loan_ID column as it is just an identifier
df = df.drop('Loan_ID', axis=1)

# Convert '3+' in Dependents to '3'
df['Dependents'] = df['Dependents'].replace('3+', '3').astype(int)

# Map Loan_Status target variable to numerical (Y=1, N=0)
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})

# Identify categorical columns for one-hot encoding (excluding Loan_Status and Dependents which are already handled)
categorical_cols_for_ohe = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']

# Apply one-hot encoding
df = pd.get_dummies(df, columns=categorical_cols_for_ohe, drop_first=True)

print("DataFrame after dropping Loan_ID, handling Dependents, mapping Loan_Status, and One-Hot Encoding:")
display(df.head())
display(df.info())

In [ ]:
# Check the distribution of the target variable
print("Distribution of Loan_Status:")
display(y.value_counts())
print("\nPercentage of Loan_Status:")
display(y.value_counts(normalize=True) * 100)

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

print("\nDistribution of y_train before SMOTE:")
display(y_train.value_counts(normalize=True) * 100)

In [ ]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE to the training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Shape of X_train after SMOTE:", X_train_smote.shape)
print("Shape of y_train after SMOTE:", y_train_smote.shape)

print("\nDistribution of y_train after SMOTE:")
display(y_train_smote.value_counts(normalize=True) * 100)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Initialize and train the Logistic Regression model
log_reg_model = LogisticRegression(random_state=42, solver='liblinear') # liblinear is good for small datasets
log_reg_model.fit(X_train_smote, y_train_smote)

# Make predictions on the test set
y_pred_log_reg = log_reg_model.predict(X_test)
y_proba_log_reg = log_reg_model.predict_proba(X_test)[:, 1]

# Evaluate the Logistic Regression model
print("Logistic Regression Model Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_log_reg):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_log_reg):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_log_reg):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_log_reg):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_log_reg):.4f}")

print("\nConfusion Matrix:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_log_reg), index=['Actual N', 'Actual Y'], columns=['Predicted N', 'Predicted Y']))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Initialize and train the Decision Tree Classifier model
dec_tree_model = DecisionTreeClassifier(random_state=42)
dec_tree_model.fit(X_train_smote, y_train_smote)

# Make predictions on the test set
y_pred_dec_tree = dec_tree_model.predict(X_test)
y_proba_dec_tree = dec_tree_model.predict_proba(X_test)[:, 1]

# Evaluate the Decision Tree model
print("Decision Tree Classifier Model Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dec_tree):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dec_tree):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_dec_tree):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_dec_tree):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_dec_tree):.4f}")

print("\nConfusion Matrix:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_dec_tree), index=['Actual N', 'Actual Y'], columns=['Predicted N', 'Predicted Y']))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and train the Random Forest Classifier model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_smote, y_train_smote)

# Make predictions on the test set
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Evaluate the Random Forest model
print("Random Forest Classifier Model Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}")

print("\nConfusion Matrix:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_rf), index=['Actual N', 'Actual Y'], columns=['Predicted N', 'Predicted Y']))

In [ ]:
# Store and compare model performance
model_performance = {
    'Logistic Regression': {
        'Accuracy': accuracy_score(y_test, y_pred_log_reg),
        'Precision': precision_score(y_test, y_pred_log_reg),
        'Recall': recall_score(y_test, y_pred_log_reg),
        'F1-Score': f1_score(y_test, y_pred_log_reg),
        'ROC-AUC': roc_auc_score(y_test, y_proba_log_reg)
    },
    'Decision Tree': {
        'Accuracy': accuracy_score(y_test, y_pred_dec_tree),
        'Precision': precision_score(y_test, y_pred_dec_tree),
        'Recall': recall_score(y_test, y_pred_dec_tree),
        'F1-Score': f1_score(y_test, y_pred_dec_tree),
        'ROC-AUC': roc_auc_score(y_test, y_proba_dec_tree)
    },
    'Random Forest': {
        'Accuracy': accuracy_score(y_test, y_pred_rf),
        'Precision': precision_score(y_test, y_pred_rf),
        'Recall': recall_score(y_test, y_pred_rf),
        'F1-Score': f1_score(y_test, y_pred_rf),
        'ROC-AUC': roc_auc_score(y_test, y_proba_rf)
    }
}

performance_df = pd.DataFrame(model_performance).T.round(4)
print("\nModel Performance Comparison:")
display(performance_df)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

# Create a figure and an axes object for the plot
fig, ax = plt.subplots(figsize=(10, 8))

# Plot ROC curve for Logistic Regression
RocCurveDisplay.from_estimator(log_reg_model, X_test, y_test, ax=ax, name='Logistic Regression')

# Plot ROC curve for Decision Tree
RocCurveDisplay.from_estimator(dec_tree_model, X_test, y_test, ax=ax, name='Decision Tree')

# Plot ROC curve for Random Forest
RocCurveDisplay.from_estimator(rf_model, X_test, y_test, ax=ax, name='Random Forest')

# Add title and labels
ax.set_title('ROC Curves for Loan Approval Models')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.grid(linestyle='--', alpha=0.7)
plt.show()

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

# Create a figure and an axes object for the plot
fig_pr, ax_pr = plt.subplots(figsize=(10, 8))

# Plot Precision-Recall curve for Logistic Regression
PrecisionRecallDisplay.from_estimator(log_reg_model, X_test, y_test, ax=ax_pr, name='Logistic Regression')

# Plot Precision-Recall curve for Decision Tree
PrecisionRecallDisplay.from_estimator(dec_tree_model, X_test, y_test, ax=ax_pr, name='Decision Tree')

# Plot Precision-Recall curve for Random Forest
PrecisionRecallDisplay.from_estimator(rf_model, X_test, y_test, ax=ax_pr, name='Random Forest')

# Add title and labels
ax_pr.set_title('Precision-Recall Curves for Loan Approval Models')
ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')
ax_pr.grid(linestyle='--', alpha=0.7)
plt.show()

## Confusion Matrices

In [ ]:
# Logistic Regression Confusion Matrix
print("Logistic Regression Confusion Matrix:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_log_reg), index=['Actual N', 'Actual Y'], columns=['Predicted N', 'Predicted Y']))

# Decision Tree Confusion Matrix
print("\nDecision Tree Confusion Matrix:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_dec_tree), index=['Actual N', 'Actual Y'], columns=['Predicted N', 'Predicted Y']))

# Random Forest Confusion Matrix
print("\nRandom Forest Confusion Matrix:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_rf), index=['Actual N', 'Actual Y'], columns=['Predicted N', 'Predicted Y']))

## Feature Importance for Random Forest Model

In [ ]:
import matplotlib.pyplot as plt

# Get feature importances from the Random Forest model
feature_importances = rf_model.feature_importances_

# Get feature names from X (original features before scaling, or scaled features if using X directly)
# Assuming X_train_smote columns represent the features after preprocessing
feature_names = X_train_smote.columns

# Create a pandas Series for easier sorting and plotting
importance_df = pd.Series(feature_importances, index=feature_names).sort_values(ascending=False)

# Plot feature importances
plt.figure(figsize=(12, 7))
importance_df.plot(kind='bar')
plt.title('Feature Importance from Random Forest Model')
plt.xlabel('Features')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Cost-Benefit Analysis: Threshold Tuning for Random Forest

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

# Get predicted probabilities for the positive class from the Random Forest model
y_proba = rf_model.predict_proba(X_test)[:, 1]

# Calculate precision, recall, and thresholds
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

# Plot the precision-recall-threshold curve
plt.figure(figsize=(12, 7))
plt.plot(thresholds, precision[:-1], label='Precision')
plt.plot(thresholds, recall[:-1], label='Recall')
plt.xlabel('Classification Threshold')
plt.ylabel('Score')
plt.title('Precision and Recall vs. Classification Threshold (Random Forest)')
plt.legend()
plt.grid(True)
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.show()


The plot above illustrates the trade-off between Precision and Recall at various classification thresholds for the Random Forest model.

*   **Higher Thresholds**: Increase precision (fewer false positives), but decrease recall (more false negatives).
*   **Lower Thresholds**: Increase recall (fewer false negatives), but decrease precision (more false positives).

To conduct a thorough cost-benefit analysis:

1.  **Quantify Costs**: Assign monetary costs to false positives (e.g., cost of a defaulting loan) and false negatives (e.g., lost revenue from a missed approved loan).
2.  **Quantify Benefits**: Assign monetary benefits to true positives (e.g., profit from a successfully repaid loan) and true negatives.
3.  **Simulate Outcomes**: For each threshold, use the confusion matrix (or predicted counts) to calculate the total cost/benefit based on your defined monetary values.
4.  **Select Optimal Threshold**: Choose the threshold that maximizes net profit or minimizes overall cost, according to your business objectives.

This plot serves as a crucial visual aid in making informed decisions about the optimal operating point for your model in a real-world scenario.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target (y)
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

# Identify numerical columns for scaling (excluding boolean columns)
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns

# Apply StandardScaler to numerical columns
scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

print("DataFrame after Feature Scaling:")
display(X.head())
display(y.head())

## Business Interpretation and Model Trade-offs

In the context of loan approval, different metrics hold varying importance depending on the business objective.

*   **Precision**: The ability of the model not to label as positive a sample that is negative (i.e., minimizing false positives). In loan approval, high precision means fewer approved loans to applicants who will default (minimizing financial losses for the bank).
*   **Recall**: The ability of the model to find all the positive samples (i.e., minimizing false negatives). High recall means approving a larger percentage of eligible applicants (maximizing business growth and inclusivity).
*   **F1-Score**: The harmonic mean of precision and recall. It's a good measure when you need to balance both precision and recall.
*   **ROC-AUC**: Represents the ability of the model to distinguish between positive and negative classes. A higher AUC indicates a better overall model performance regardless of the classification threshold.

### Model Comparison:

1.  **Logistic Regression:**
    *   **Pros:** Good balance of precision and recall, decent ROC-AUC. It's interpretable, which is valuable for regulated industries like finance. Its Recall (0.9059) is high, meaning it's good at identifying actual loan defaulters. It has the highest ROC-AUC among the three, suggesting good discrimination ability.
    *   **Cons:** Accuracy is slightly lower than Random Forest.

2.  **Decision Tree:**
    *   **Pros:** Easy to understand and visualize. It has a high Precision (0.8169) which is good to avoid false positives (approving bad loans).
    *   **Cons:** Significantly lower Accuracy, Recall, F1-Score, and ROC-AUC compared to the other two models. It is prone to overfitting if not properly pruned.

3.  **Random Forest:**
    *   **Pros:** Highest Accuracy (0.8293) and F1-Score (0.8800), and tied for highest Recall (0.9059). Generally robust against overfitting. Very good at identifying actual loan defaulters. Its Precision is also good.
    *   **Cons:** Lower ROC-AUC than Logistic Regression. Less interpretable than Logistic Regression or a single Decision Tree.

### Trade-offs and Recommendation:

*   **If minimizing financial losses from defaults is the absolute priority (high cost of false positives):** A model with higher precision would be preferred. Random Forest has a slightly higher precision than Logistic Regression. However, Logistic Regression's overall balance and interpretability make it a strong contender.

*   **If maximizing the number of approved eligible loans is crucial (high cost of false negatives):** Both Logistic Regression and Random Forest excel with very high Recall scores (0.9059). Both models are good at identifying applicants who *should* be approved.

Considering the performance metrics, **Random Forest** appears to be the best performing model overall in terms of Accuracy, F1-Score, and Recall. While Logistic Regression has a slightly better ROC-AUC and is more interpretable, Random Forest's superior F1-Score and Accuracy suggest better overall predictive power. For a business context, Random Forest would likely lead to a better balance of identifying good loan applicants while maintaining reasonable control over potential defaults.

### Suggested Deployment Threshold:

The default classification threshold for most models is 0.5. This means if the predicted probability of loan approval is > 0.5, the loan is approved. However, this threshold can be adjusted based on business risk appetite.

*   **Current Threshold (0.5):** The model performance metrics discussed above are based on this default threshold.

*   **Adjusting the Threshold:**
    *   **Increase threshold (>0.5):** This would make the model more conservative, leading to fewer approved loans but also fewer defaults (higher precision, lower recall). This might be desired if the bank wants to significantly reduce risk.
    *   **Decrease threshold (<0.5):** This would make the model more lenient, leading to more approved loans but potentially more defaults (lower precision, higher recall). This might be considered if the bank is looking for aggressive growth and is willing to accept a higher risk.

**Recommendation:** Given that Random Forest has the highest F1-Score and a high Recall, indicating a good balance, we could start with the default threshold of **0.5**. However, it is crucial to perform a **cost-benefit analysis** specific to the bank's operational costs, potential losses from defaults, and profits from approved loans. This analysis would involve plotting Precision-Recall curves or ROC curves to visualize the trade-offs at different thresholds and selecting the threshold that optimizes business outcomes.

## Conclusion and Report Summary

This notebook demonstrates the development of a supervised machine learning model to predict loan approval based on borrower features. The process involved several key stages:

1.  **Data Loading and Initial Inspection**: The `loan_prediction.csv` dataset was loaded, and its structure, data types, and missing values were initially examined.

2.  **Missing Value Handling**: Missing values in categorical columns (`Credit_History`, `Self_Employed`, `Loan_Amount_Term`, `Gender`, `Married`, `Dependents`) were imputed using the mode, while the numerical `LoanAmount` was imputed using the median to maintain robustness against outliers.

3.  **Categorical Variable Encoding**: The `Loan_ID` was dropped as it's an identifier. The `Dependents` column's '3+' was converted to '3' and cast to integer. The target variable `Loan_Status` was mapped to numerical values (1 for 'Y', 0 for 'N'). Other nominal categorical features were then one-hot encoded.

4.  **Feature Scaling**: Numerical features were scaled using `StandardScaler` to ensure that no single feature dominates the model due to its scale.

5.  **Class Imbalance Handling**: The target variable `Loan_Status` exhibited class imbalance. To address this, the data was split into training and testing sets using stratification, and SMOTE (Synthetic Minority Over-sampling Technique) was applied to the training data to balance the classes.

6.  **Model Training and Evaluation**: Three different classification models were trained and evaluated:
    *   **Logistic Regression**
    *   **Decision Tree Classifier**
    *   **Random Forest Classifier**

    The models were evaluated using key metrics: Accuracy, Precision, Recall, F1-Score, and ROC-AUC.

    **Model Performance Comparison:**

    | Model                 | Accuracy | Precision | Recall | F1-Score | ROC-AUC |
    |:----------------------|:---------|:----------|:-------|:---------|:--------|
    | Logistic Regression   | 0.8130   | 0.8370    | 0.9059 | 0.8701   | 0.8220  |
    | Decision Tree         | 0.6748   | 0.8169    | 0.6824 | 0.7436   | 0.6701  |
    | Random Forest         | 0.8293   | 0.8556    | 0.9059 | 0.8800   | 0.7565  |

    **Business Interpretation and Trade-offs:**

    *   **Random Forest** showed the best overall performance, achieving the highest Accuracy (0.8293) and F1-Score (0.8800), and a strong Recall (0.9059). This suggests it is highly effective at identifying loan approvals while maintaining good balance across precision and recall. Its primary trade-off is its reduced interpretability compared to Logistic Regression.

    *   **Logistic Regression** offered a competitive performance with good balance and the highest ROC-AUC (0.8220), making it a valuable option due to its inherent interpretability, which is often crucial in financial applications.

    *   **Decision Tree** performed noticeably weaker than the other models across most metrics, indicating it is less suitable for this particular problem without significant tuning.

    **Suggested Deployment Threshold:**

    For deployment, it is recommended to start with a classification probability threshold of **0.5**. This threshold, combined with the **Random Forest** model, provides a robust balance between approving eligible applicants (high recall) and minimizing defaults (good precision).

    However, the optimal threshold should be determined through a **detailed cost-benefit analysis** that considers the financial implications of false positives (approving a defaulting loan) versus false negatives (rejecting a creditworthy applicant). Adjusting the threshold above 0.5 would increase precision (fewer defaults) at the cost of recall (fewer approvals), while lowering it would increase recall (more approvals) at the cost of precision (more defaults). This analysis, ideally using precision-recall curves, will fine-tune the model's application to specific business objectives and risk appetite.